# Snapshot Wisconsin: head-visibility gold set

This notebook creates a manually reviewed gold set of 250 deer images. It does not train or run YOLO, CNN, or Qwen.

For every image, choose one label:

- `V`: deer head visible
- `N`: deer head not visible
- `U`: uncertain / needs a second reviewer

Labels are saved after every batch so progress is not lost if the notebook stops.

## 1. Configure the data paths

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import clear_output, display

# Change DATA_ROOT if your CSV and photo folder are stored elsewhere.
DATA_ROOT = Path.cwd()
CSV_PATH = DATA_ROOT / 'Snapshot_WI-Oh_Deer_data-v1.csv'
PHOTOS_DIR = DATA_ROOT / 'Snapshot_WI-Oh_Deer_Photos' / 'Deer'
OUTPUT_DIR = DATA_ROOT / 'gold_set'
MANIFEST_PATH = OUTPUT_DIR / 'gold_set_manifest.csv'
ANNOTATIONS_PATH = OUTPUT_DIR / 'gold_set_annotations.csv'

TARGET_COUNT = 250       # Keep this between 200 and 300.
BATCH_SIZE = 25
RANDOM_SEED = 42
REVIEWER = 'manual-reviewer'

if not 200 <= TARGET_COUNT <= 300:
    raise ValueError('TARGET_COUNT must be between 200 and 300.')
if not CSV_PATH.exists():
    raise FileNotFoundError(f'CSV not found: {CSV_PATH}')
if not PHOTOS_DIR.exists():
    raise FileNotFoundError(f'Photo directory not found: {PHOTOS_DIR}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(CSV_PATH)
if 'filename' not in df.columns:
    raise ValueError("The CSV must contain a 'filename' column.")
df = df.drop_duplicates(subset=['filename']).copy()
df['image_path'] = df['filename'].map(lambda name: str(PHOTOS_DIR / str(name)))
df = df[df['image_path'].map(lambda path: Path(path).exists())].reset_index(drop=True)

print(f'Usable images: {len(df):,}')
print(f'Gold-set target: {TARGET_COUNT}')

## 2. Select and save a reproducible sample

The sample is selected once with a fixed seed. Re-running this cell reuses the existing manifest, so the image set does not change after annotation has started.

In [ ]:
if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)
    print(f'Loaded existing manifest: {len(manifest)} images')
else:
    if len(df) < TARGET_COUNT:
        raise ValueError(f'Only {len(df)} usable images are available; need {TARGET_COUNT}.')
    manifest = df.sample(n=TARGET_COUNT, random_state=RANDOM_SEED).copy()
    manifest.insert(0, 'sample_id', np.arange(1, len(manifest) + 1))
    manifest['image_path'] = manifest['image_path'].astype(str)
    manifest[['sample_id', 'filename', 'image_path']].to_csv(MANIFEST_PATH, index=False)
    print(f'Created manifest: {MANIFEST_PATH}')

manifest = manifest[['sample_id', 'filename', 'image_path']].copy()
if len(manifest) < 200 or len(manifest) > 300:
    raise ValueError(f'Manifest contains {len(manifest)} images; expected 200-300.')
display(manifest.head())

## 3. Create or resume the annotation file

In [ ]:
annotation_columns = [
    'sample_id', 'filename', 'head_visibility', 'confidence',
    'notes', 'reviewer', 'reviewed_at'
]
if ANNOTATIONS_PATH.exists():
    annotations = pd.read_csv(ANNOTATIONS_PATH)
    annotations = annotations.reindex(columns=annotation_columns)
    print(f'Resuming annotations: {len(annotations)} rows')
else:
    annotations = manifest[['sample_id', 'filename']].copy()
    annotations['head_visibility'] = ''
    annotations['confidence'] = ''
    annotations['notes'] = ''
    annotations['reviewer'] = ''
    annotations['reviewed_at'] = ''
    annotations.to_csv(ANNOTATIONS_PATH, index=False)
    print(f'Created annotation file: {ANNOTATIONS_PATH}')

annotations = annotations.set_index('sample_id').reindex(manifest['sample_id']).reset_index()
annotations['filename'] = manifest['filename'].values
annotations.to_csv(ANNOTATIONS_PATH, index=False)
print('Unlabeled:', (annotations['head_visibility'].fillna('') == '').sum())
display(annotations.head())

## 4. Review images in batches

Each batch displays 25 images. After reviewing the grid, enter one label per image in order, separated by commas. Example: `V,V,N,U,V,...`.

You may enter `V`, `N`, or `U` in either upper- or lowercase. A confidence value of `1`, `2`, or `3` is optional (`3` = high confidence). Pressing Enter without labels skips the batch.

In [ ]:
VALID_LABELS = {'V': 'visible', 'N': 'not_visible', 'U': 'uncertain'}

def show_batch(batch):
    columns = 5
    rows = math.ceil(len(batch) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(18, rows * 3.2))
    axes = np.atleast_1d(axes).ravel()
    for axis, (_, row) in zip(axes, batch.iterrows()):
        try:
            image = Image.open(row['image_path']).convert('RGB')
            axis.imshow(image)
        except Exception as exc:
            axis.text(0.5, 0.5, f'Could not load image\n{exc}', ha='center', va='center')
        axis.set_title(f"#{int(row['sample_id'])}\\n{row['filename']}", fontsize=7)
        axis.axis('off')
    for axis in axes[len(batch):]:
        axis.axis('off')
    plt.tight_layout()
    display(fig)
    plt.close(fig)

def save_annotations():
    annotations.to_csv(ANNOTATIONS_PATH, index=False)

for start in range(0, len(manifest), BATCH_SIZE):
    batch_ids = manifest.iloc[start:start + BATCH_SIZE]['sample_id'].tolist()
    batch = manifest[manifest['sample_id'].isin(batch_ids)].copy()
    pending = annotations[annotations['sample_id'].isin(batch_ids)]['head_visibility'].fillna('')
    if (pending != '').all():
        continue

    clear_output(wait=True)
    print(f'Batch {start // BATCH_SIZE + 1} / {math.ceil(len(manifest) / BATCH_SIZE)}')
    show_batch(batch)
    raw_labels = input(f'Labels for {len(batch)} images (V/N/U, comma-separated): ').strip()
    if not raw_labels:
        print('Skipped. Re-run this cell to review the batch later.')
        continue

    labels = [label.strip().upper() for label in raw_labels.split(',')]
    if len(labels) != len(batch) or any(label not in VALID_LABELS for label in labels):
        raise ValueError(f'Expected exactly {len(batch)} labels, each V, N, or U; received {labels}.')

    confidence_raw = input('Optional confidence values (1/2/3, comma-separated; blank = empty): ').strip()
    confidences = [value.strip() for value in confidence_raw.split(',')] if confidence_raw else [''] * len(batch)
    if len(confidences) != len(batch) or any(value not in {'', '1', '2', '3'} for value in confidences):
        raise ValueError(f'Expected {len(batch)} confidence values, each blank, 1, 2, or 3.')

    notes = input('Optional note for this batch (blank = none): ').strip()
    now = datetime.now(timezone.utc).isoformat()
    for sample_id, label, confidence in zip(batch_ids, labels, confidences):
        row_index = annotations.index[annotations['sample_id'] == sample_id][0]
        annotations.loc[row_index, 'head_visibility'] = VALID_LABELS[label]
        annotations.loc[row_index, 'confidence'] = confidence
        annotations.loc[row_index, 'notes'] = notes
        annotations.loc[row_index, 'reviewer'] = REVIEWER
        annotations.loc[row_index, 'reviewed_at'] = now
    save_annotations()
    print(f'Saved {len(batch)} annotations to {ANNOTATIONS_PATH}')

clear_output(wait=True)
print('Review loop finished. Re-run this cell if any batches were skipped.')
print(annotations['head_visibility'].replace('', pd.NA).value_counts(dropna=False))

## 5. Validate and summarize the gold set

Do not use the gold set for model training until all `uncertain` rows have been resolved or explicitly kept as an excluded review set.

In [ ]:
annotations = pd.read_csv(ANNOTATIONS_PATH)
allowed = {'visible', 'not_visible', 'uncertain'}
invalid = annotations.loc[
    annotations['head_visibility'].fillna('').ne('') &
    ~annotations['head_visibility'].isin(allowed)
]
if len(invalid):
    raise ValueError(f'Invalid labels found:\n{invalid}')

counts = annotations['head_visibility'].replace('', pd.NA).value_counts(dropna=False)
print(f'Output: {ANNOTATIONS_PATH}')
print(f'Completed: {annotations["head_visibility"].isin(allowed).sum()} / {len(annotations)}')
print(counts)
display(annotations)
